In [1]:
import sys
sys.path.append('../')

import torch
import pandas as pd
import optuna
import warnings
from copy import deepcopy
from torch.utils.data import DataLoader
from src.mypackage.data_preparation import prepare_statistical_data
from src.mypackage.torch_dataset import EnergyDataset
from src.mypackage.trainer import Trainer
from src.mypackage.rnn_models import RNNModel
from src.mypackage.forecasting import dnn_forecast
from src.mypackage.evaluation import print_metrics, get_true_values
from src.mypackage.visualization import plot_forecast
from src.mypackage.utils import set_seed, SEED

warnings.filterwarnings("ignore")
set_seed(SEED)

In [2]:
# ========== データ読み込みとデータセット準備 ==========
df = pd.read_csv("../data/raw/PJME_hourly.csv")
_, tmp = prepare_statistical_data(df)
timesteps = tmp["Datetime"].copy()
del tmp
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

SEQ_LEN = 168
PRED_LEN = 24
SHIFT = 24
BATCH_SIZE = 32

# ========== データセット作成 ==========
dataset = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=PRED_LEN, mode="re-train")
retrain_dataset = deepcopy(dataset)
train_dataset = deepcopy(dataset)
train_dataset.mode_switch("train")
valid_dataset = deepcopy(dataset)
valid_dataset.mode_switch("val")
test_dataset = deepcopy(dataset)
test_dataset.mode_switch("test")

# DataLoader作成
retrain_loader = DataLoader(retrain_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Whole dataset size: {len(retrain_dataset)}")
print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Input features: {train_dataset[0][0].shape[1]}")


Dataset shape: (145366, 2)
Columns: ['Datetime', 'PJME_MW']


Whole dataset size: 5654
Train dataset size: 5289
Val dataset size: 358
Test dataset size: 358
Input features: 19


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ========== ハイパーパラメータ設定 ==========
# 固定
NUM_EPOCHS = 100
INPUT_SIZE = train_dataset[0][0].shape[1]
EARLY_STOPPING_PATIENCE = 10
OUTPUT_SIZE = PRED_LEN

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    hidden_size = 2 ** trial.suggest_int("hidden_size", 4, 9)
    num_layers = trial.suggest_int("num_layers", 1, 4)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    model_type = trial.suggest_categorical("model_type", ["RNN", "LSTM", "GRU"])

    model = RNNModel(input_size=INPUT_SIZE, 
                     hidden_size=hidden_size, 
                     num_layers=num_layers, 
                     output_size=OUTPUT_SIZE, 
                     dropout=dropout,
                     rnn_type=model_type).to(device)

    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        checkpoint_dir="../models/rnn_checkpoints"
    )
    history = trainer.train(num_epochs=NUM_EPOCHS, verbose=0)
    best_iteration = trainer.early_stopping.best_iteration
    trial.set_user_attr("best_iteration", best_iteration)
    return min(history['val_loss'])

Using device: cuda


In [7]:
# パラメータ探索の実行
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50)
print(study.best_params)

[I 2026-07-17 08:59:18,289] A new study created in memory with name: no-name-5c24bc90-b414-4093-801e-6d4afd2c7e40
[I 2026-07-17 09:01:22,721] Trial 0 finished with value: 0.10017266233141224 and parameters: {'learning_rate': 0.0001329291894316216, 'weight_decay': 0.0007114476009343421, 'hidden_size': 8, 'num_layers': 3, 'dropout': 0.1624074561769746, 'model_type': 'GRU'}. Best is trial 0 with value: 0.10017266233141224.


Checkpoint loaded from ../models/rnn_checkpoints/best_model.pt


[I 2026-07-17 09:01:46,550] Trial 1 finished with value: 0.13122985636194548 and parameters: {'learning_rate': 0.0006358358856676254, 'weight_decay': 0.000133112160807369, 'hidden_size': 4, 'num_layers': 4, 'dropout': 0.4329770563201687, 'model_type': 'RNN'}. Best is trial 0 with value: 0.10017266233141224.


Checkpoint loaded from ../models/rnn_checkpoints/best_model.pt


[I 2026-07-17 09:02:22,293] Trial 2 finished with value: 0.10133366814504068 and parameters: {'learning_rate': 8.17949947521167e-05, 'weight_decay': 3.752055855124284e-05, 'hidden_size': 6, 'num_layers': 2, 'dropout': 0.34474115788895177, 'model_type': 'GRU'}. Best is trial 0 with value: 0.10017266233141224.


...

Checkpoint loaded from ../models/rnn_checkpoints/best_model.pt


[I 2026-07-17 09:44:45,404] Trial 48 finished with value: 0.09125215436021487 and parameters: {'learning_rate': 0.0003799136160779007, 'weight_decay': 0.0002005606356760958, 'hidden_size': 9, 'num_layers': 1, 'dropout': 0.4294562286936676, 'model_type': 'GRU'}. Best is trial 47 with value: 0.08593229483813047.


Checkpoint loaded from ../models/rnn_checkpoints/best_model.pt


[I 2026-07-17 09:45:39,332] Trial 49 finished with value: 0.09458969036738078 and parameters: {'learning_rate': 8.476125657881433e-05, 'weight_decay': 0.0005303948385821847, 'hidden_size': 8, 'num_layers': 1, 'dropout': 0.10826207854925542, 'model_type': 'GRU'}. Best is trial 47 with value: 0.08593229483813047.


Checkpoint loaded from ../models/rnn_checkpoints/best_model.pt
{'learning_rate': 0.00043239443227223684, 'weight_decay': 0.0003819016709117818, 'hidden_size': 8, 'num_layers': 1, 'dropout': 0.4277293174778006, 'model_type': 'GRU'}


In [8]:
# モデルの再学習（1日予測）
model = RNNModel(input_size=INPUT_SIZE,
                 hidden_size=2**study.best_params["hidden_size"], 
                 num_layers=study.best_params["num_layers"], 
                 output_size=OUTPUT_SIZE, 
                 dropout=study.best_params["dropout"],
                 rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=retrain_loader,
                  test_loader=test_loader,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

Epoch [2/22] - Train Loss: 0.138820, Val Loss: nan - Train MAE: 0.285577, Val MAE: nan
Epoch [4/22] - Train Loss: 0.121315, Val Loss: nan - Train MAE: 0.264401, Val MAE: nan
Epoch [6/22] - Train Loss: 0.112748, Val Loss: nan - Train MAE: 0.254827, Val MAE: nan
Epoch [8/22] - Train Loss: 0.112205, Val Loss: nan - Train MAE: 0.255247, Val MAE: nan
Epoch [10/22] - Train Loss: 0.109623, Val Loss: nan - Train MAE: 0.251240, Val MAE: nan
Epoch [12/22] - Train Loss: 0.104971, Val Loss: nan - Train MAE: 0.245065, Val MAE: nan
Epoch [14/22] - Train Loss: 0.103147, Val Loss: nan - Train MAE: 0.243515, Val MAE: nan
Epoch [16/22] - Train Loss: 0.101261, Val Loss: nan - Train MAE: 0.240507, Val MAE: nan
Epoch [18/22] - Train Loss: 0.103139, Val Loss: nan - Train MAE: 0.244001, Val MAE: nan
Epoch [20/22] - Train Loss: 0.109295, Val Loss: nan - Train MAE: 0.252098, Val MAE: nan
Epoch [22/22] - Train Loss: 0.102303, Val Loss: nan - Train MAE: 0.241320, Val MAE: nan
No best model checkpoint found; save

In [9]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_checkpoints/day_best_model.pth")
trainer.save_config("../models/rnn_checkpoints/day_best_model_config.json")


Test Metrics:
  Test Loss (MSE): 0.098480
  Test MAE: 0.219652
  Test RMSE: 0.301393
Checkpoint saved to ../models/rnn_checkpoints/day_best_model.pth
Configuration saved to ../models/rnn_checkpoints/day_best_model_config.json


In [10]:
# 1週間予測用のデータセット
dataset_week = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=7*PRED_LEN, mode="re-train")
test_dataset_week = deepcopy(dataset_week)
test_dataset_week.mode_switch("test")

train_loader_week = DataLoader(dataset_week, batch_size=BATCH_SIZE, shuffle=True)
test_loader_week = DataLoader(test_dataset_week, batch_size=BATCH_SIZE, shuffle=False)

In [12]:
# モデルの再学習（1週間予測）
model = RNNModel(input_size=INPUT_SIZE,
                 hidden_size=2**study.best_params["hidden_size"], 
                 num_layers=study.best_params["num_layers"], 
                 output_size=7*PRED_LEN,  # 1週間予測のため出力サイズを変更
                 dropout=study.best_params["dropout"],
                 rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_week,
                  test_loader=test_loader_week,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

Epoch [2/22] - Train Loss: 0.276712, Val Loss: nan - Train MAE: 0.400219, Val MAE: nan
Epoch [4/22] - Train Loss: 0.254383, Val Loss: nan - Train MAE: 0.380108, Val MAE: nan
Epoch [6/22] - Train Loss: 0.240399, Val Loss: nan - Train MAE: 0.367678, Val MAE: nan
Epoch [8/22] - Train Loss: 0.239660, Val Loss: nan - Train MAE: 0.366932, Val MAE: nan
Epoch [10/22] - Train Loss: 0.233739, Val Loss: nan - Train MAE: 0.362740, Val MAE: nan
Epoch [12/22] - Train Loss: 0.229642, Val Loss: nan - Train MAE: 0.358166, Val MAE: nan
Epoch [14/22] - Train Loss: 0.227754, Val Loss: nan - Train MAE: 0.357523, Val MAE: nan
Epoch [16/22] - Train Loss: 0.225914, Val Loss: nan - Train MAE: 0.355466, Val MAE: nan
Epoch [18/22] - Train Loss: 0.222216, Val Loss: nan - Train MAE: 0.353219, Val MAE: nan
Epoch [20/22] - Train Loss: 0.224403, Val Loss: nan - Train MAE: 0.354827, Val MAE: nan
Epoch [22/22] - Train Loss: 0.221807, Val Loss: nan - Train MAE: 0.352133, Val MAE: nan
No best model checkpoint found; save

In [13]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_checkpoints/week_best_model.pth")
trainer.save_config("../models/rnn_checkpoints/week_best_model_config.json")


Test Metrics:
  Test Loss (MSE): 0.263101
  Test MAE: 0.378883
  Test RMSE: 0.486181
Checkpoint saved to ../models/rnn_checkpoints/week_best_model.pth
Configuration saved to ../models/rnn_checkpoints/week_best_model_config.json


In [14]:
# 1か月予測用のデータセット
dataset_month = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=30*PRED_LEN, mode="re-train")
test_dataset_month = deepcopy(dataset_month)
test_dataset_month.mode_switch("test")

train_loader_month = DataLoader(dataset_month, batch_size=BATCH_SIZE, shuffle=True)
test_loader_month = DataLoader(test_dataset_month, batch_size=BATCH_SIZE, shuffle=False)

In [15]:
# モデルの再学習（1か月予測）
model = RNNModel(input_size=INPUT_SIZE,
                 hidden_size=2**study.best_params["hidden_size"], 
                 num_layers=study.best_params["num_layers"], 
                 output_size=30*PRED_LEN,  # 1か月予測のため出力サイズを変更
                 dropout=study.best_params["dropout"],
                 rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_month,
                  test_loader=test_loader_month,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

Epoch [2/22] - Train Loss: 0.305794, Val Loss: nan - Train MAE: 0.422287, Val MAE: nan
Epoch [4/22] - Train Loss: 0.284012, Val Loss: nan - Train MAE: 0.404129, Val MAE: nan
Epoch [6/22] - Train Loss: 0.271789, Val Loss: nan - Train MAE: 0.394821, Val MAE: nan
Epoch [8/22] - Train Loss: 0.264784, Val Loss: nan - Train MAE: 0.389411, Val MAE: nan
Epoch [10/22] - Train Loss: 0.262757, Val Loss: nan - Train MAE: 0.387315, Val MAE: nan
Epoch [12/22] - Train Loss: 0.259160, Val Loss: nan - Train MAE: 0.384341, Val MAE: nan
Epoch [14/22] - Train Loss: 0.256656, Val Loss: nan - Train MAE: 0.382661, Val MAE: nan
Epoch [16/22] - Train Loss: 0.255289, Val Loss: nan - Train MAE: 0.380931, Val MAE: nan
Epoch [18/22] - Train Loss: 0.253279, Val Loss: nan - Train MAE: 0.379247, Val MAE: nan
Epoch [20/22] - Train Loss: 0.249881, Val Loss: nan - Train MAE: 0.377026, Val MAE: nan
Epoch [22/22] - Train Loss: 0.252439, Val Loss: nan - Train MAE: 0.378869, Val MAE: nan
No best model checkpoint found; save

In [16]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_checkpoints/month_best_model.pth")
trainer.save_config("../models/rnn_checkpoints/month_best_model_config.json")


Test Metrics:
  Test Loss (MSE): 0.334951
  Test MAE: 0.436047
  Test RMSE: 0.554523
Checkpoint saved to ../models/rnn_checkpoints/month_best_model.pth
Configuration saved to ../models/rnn_checkpoints/month_best_model_config.json


In [17]:
# 1年予測用のデータセット
dataset_year = EnergyDataset(df, seq_len=SEQ_LEN, shift=SHIFT, pred_len=365*PRED_LEN, mode="re-train")
test_dataset_year = deepcopy(dataset_year)
test_dataset_year.mode_switch("test")

train_loader_year = DataLoader(dataset_year, batch_size=BATCH_SIZE, shuffle=True)
test_loader_year = DataLoader(test_dataset_year, batch_size=BATCH_SIZE, shuffle=False)

In [18]:
# モデルの再学習（1年予測）
model = RNNModel(input_size=INPUT_SIZE,
                 hidden_size=2**study.best_params["hidden_size"], 
                 num_layers=study.best_params["num_layers"], 
                 output_size=365*PRED_LEN,  # 1年予測のため出力サイズを変更
                 dropout=study.best_params["dropout"],
                 rnn_type=study.best_params["model_type"]).to(device)

trainer = Trainer(model=model,
                  train_loader=train_loader_year,
                  test_loader=test_loader_year,
                  device=device,
                  learning_rate=study.best_params["learning_rate"],
                  weight_decay=study.best_params["weight_decay"],
                  early_stopping_patience=EARLY_STOPPING_PATIENCE,
                  checkpoint_dir="../models/rnn_checkpoints")

history = trainer.train(num_epochs=study.best_trial.user_attrs["best_iteration"], verbose=True)

Epoch [2/22] - Train Loss: 0.367504, Val Loss: nan - Train MAE: 0.463366, Val MAE: nan
Epoch [4/22] - Train Loss: 0.318179, Val Loss: nan - Train MAE: 0.429840, Val MAE: nan
Epoch [6/22] - Train Loss: 0.305740, Val Loss: nan - Train MAE: 0.420759, Val MAE: nan
Epoch [8/22] - Train Loss: 0.301588, Val Loss: nan - Train MAE: 0.417838, Val MAE: nan
Epoch [10/22] - Train Loss: 0.297876, Val Loss: nan - Train MAE: 0.414594, Val MAE: nan
Epoch [12/22] - Train Loss: 0.292377, Val Loss: nan - Train MAE: 0.410141, Val MAE: nan
Epoch [14/22] - Train Loss: 0.286388, Val Loss: nan - Train MAE: 0.405173, Val MAE: nan
Epoch [16/22] - Train Loss: 0.286477, Val Loss: nan - Train MAE: 0.405470, Val MAE: nan
Epoch [18/22] - Train Loss: 0.283621, Val Loss: nan - Train MAE: 0.403165, Val MAE: nan
Epoch [20/22] - Train Loss: 0.283323, Val Loss: nan - Train MAE: 0.402983, Val MAE: nan
Epoch [22/22] - Train Loss: 0.283918, Val Loss: nan - Train MAE: 0.403559, Val MAE: nan
No best model checkpoint found; save

In [19]:
# ========== テストデータで評価 ==========
test_loss, test_mae, test_rmse = trainer.test()
print("\nTest Metrics:")
print(f"  Test Loss (MSE): {test_loss:.6f}")
print(f"  Test MAE: {test_mae:.6f}")
print(f"  Test RMSE: {test_rmse:.6f}")

# チェックポイントとコンフィグの保存
trainer.save_checkpoint("../models/rnn_checkpoints/year_best_model.pth")
trainer.save_config("../models/rnn_checkpoints/year_best_model_config.json")

ZeroDivisionError: float division by zero

In [21]:
# チェックポイントの読み込みと再学習
checkpoint_day = torch.load("../models/rnn_checkpoints/day_best_model.pth", map_location=device)
params_day = checkpoint_day["model_config"]
model_day = RNNModel(**params_day).to(device)
model_day.load_state_dict(checkpoint_day["model_state_dict"])

checkpoint_week = torch.load("../models/rnn_checkpoints/week_best_model.pth", map_location=device)
params_week = checkpoint_week["model_config"]
model_week = RNNModel(**params_week).to(device)
model_week.load_state_dict(checkpoint_week["model_state_dict"])

checkpoint_month = torch.load("../models/rnn_checkpoints/month_best_model.pth", map_location=device)
params_month = checkpoint_month["model_config"]
model_month = RNNModel(**params_month).to(device)
model_month.load_state_dict(checkpoint_month["model_state_dict"])

checkpoint_year = torch.load("../models/rnn_checkpoints/year_best_model.pth", map_location=device)
params_year = checkpoint_year["model_config"]
model_year = RNNModel(**params_year).to(device)
model_year.load_state_dict(checkpoint_year["model_state_dict"])

<All keys matched successfully>

In [22]:
y_pred_day = dnn_forecast(model_day, test_loader, device)
y_pred_week = dnn_forecast(model_week, test_loader_week, device)
y_pred_month = dnn_forecast(model_month, test_loader_month, device)
y_pred_year = dnn_forecast(model_year, test_loader_year, device)

y_true = get_true_values(test_loader)

AttributeError: 'DataLoader' object has no attribute 'inverse_transform'

In [ ]:
print_metrics(y_true, y_pred_day, "RNN Forecast (1 Day)")
print_metrics(y_true, y_pred_week, "RNN Forecast (1 Week)")
print_metrics(y_true, y_pred_month, "RNN Forecast (1 Month)")
print_metrics(y_true, y_pred_year, "RNN Forecast (1 Year)")

In [ ]:
plot_forecast(y_true, y_pred_day, timesteps, "RNN Forecast (1 Day)")

In [ ]:
plot_forecast(y_true, y_pred_week, timesteps, "RNN Forecast (1 Week)")

In [ ]:
plot_forecast(y_true, y_pred_month, timesteps, "RNN Forecast (1 Month)")

In [ ]:
plot_forecast(y_true, y_pred_year, timesteps, "RNN Forecast (1 Year)")